# Notes

<u>System questions/notes:</u>
* What is a cluster?
* Collection - do we need partitions?
* The schema seems automatically determined based on the attributes and fields of the Documents. AutoID and dynamic field are N/A (`False` and `False`) when using Milvus with Haystack.
    ```python
    assert doc.id == res[0]["id"] # primary field
    assert doc.embedding == res[0]["vector"] # embedding
    assert doc.content == res[0]["text"] # text
    assert doc.meta["source_id"] == res[0]["source_id"] # autogenerated file ID
    assert doc.meta["page_number"] == res[0]["page_number"] # file page number
    assert doc.meta["split_id"] == res[0]["split_id"] # split index
    assert doc.meta["split_idx_start"] == res[0]["split_idx_start"] # not sure what this means
    assert doc.meta["file_path"] == res[0]["file_path"] # file
    ```
    * Primary field needs to be unique, but if I'm adding documents to an already existing collection, how do I know "id" and "source_id" will be unique?
    * doc.meta["_split_overlap"] was discarded with the error `has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.`. I think this is because doc.meta["_split_overlap"] is `list[dict]`. This might be important for retrieval.
    * Understand what the metadata means and how it's being generated.
    * Is there any additional metadata that we might want to add that isn't autogenerated? Can we add additional metadata in the pipeline? Or do we need to predefine metadata in the vector DB, before using the pipeline to write to it? Example metadata:
        * **tag**
        * Course / course number / course name
        * Lecturer
        * Year / semester
        * Lecture number or lecture title
    * Is it possible for different file types to have different metadata? If so, how can we handle that?
* Figure out how AUTOINDEX works; understand all the different indexes and metrics
    * What kind of metric should we use? This depends on whether the SentenceTransformer embeddings are normalized. What's the best practice?
    * Can I change the metric after the collection is created? I'm guessing that yes, I can, but the indexing will have to be rerun since the indexing depends on the metric.
    * You should also index by tag. Zilliz uses **TRIE** for integers and **STL_SORT** for strings. Can I add an index after the collection is already created?
* Vector fields
    * Should we have any kind of metadata embedding?
    * Multimodal embedding for images?
    * Utilize both dense and sparse embeddings, and search both using `hybrid_search()`?
* Pipelines
    * Indexing - this should be relatively straightforward
    * Tagging - get the new IDs that were indexed into the vector DB; pass them through the LLM to generate tags; add tag metadata to the new entities

<u>General questions:</u>
* If we know the course the learning material is from, why can't we get the tag directly based on the course/learning track? Guess: Courses may cover many subdisciplines; courses are not necessarily rigidly only one discipline.

<u>Zilliz notes:</u>
* Quickstart notes
    * The insert operations are asynchronous, and conducting a search immediately after data insertions may result in empty result set. To avoid this, you are advised to wait for a few seconds.
    * Searches are semantic searches (client.search), but you can also apply scalar field filters. Queries (client.query) are based on scalar filters only. You can also directly retrieve entities by their ID using client.get (instead of using query).
* Collection notes
    * You need to load a collection into memory to search and query it. This means loading the index files and the raw data of the fields.
    * A collection cannot be loaded without an index file
    * To reduce memory usage and improve search performance, you can specify which fields you want to load (instead of all fields)
        * Only these fields may be used for filtering and as outputs in search and query
        * You should always include the primary field and at least one vector field
    * Entities inserted after a collection load are automatically indexed and loaded
* Indexing notes
    * Recommended to create indexes for both vector field(s) and scalar field(s) that are frequently accessed.
    * Vector field indexes are for semantic search; scalar field indexes are for metadata filtering.
    * You can create up to one index per field in a collection.
    * For vector field indexes, Zilliz Cloud supports AUTOINDEX (https://docs.zilliz.com/docs/autoindex-explained)
        * Performance-optimized and capacity-optimized clusters require different approaches to indexing - AUTOINDEX takes care of that
        * Improved performance via SIMD, data graphing and cropping, and dynamic quantization
        * AUTOINDEX automatically chooses search parameters to trade off between recall and performance. Search params only has 1 parameter: level. Higher level = higher recall, but possibly slower search. Level defaults to 1 and ranges from 1 to 10. Default value = 90% recall. `enable_recall_calculation`?

# Zilliz/Milvus API

In [44]:
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv
import os

load_dotenv()

True

In [45]:
client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

# client = MilvusClient(
#     uri="milvus.db"
# )

In [24]:
client.close()

## Collections

In [46]:
# View collections

coll_name = client.list_collections()[0]
# print(coll_name)
# print("-"*20)
coll_description = client.describe_collection(coll_name)
for key, value in coll_description.items():
    if type(value) == list:
        print(f"{key}:")
        for v in value:
            print(v)
    else:
        print(f"{key}: {value}")
    print("-"*20)

collection_name: HaystackCollection
--------------------
auto_id: False
--------------------
num_shards: 1
--------------------
description: Test collection
--------------------
fields:
{'field_id': 100, 'name': 'file_path', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 101, 'name': 'source_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 102, 'name': 'page_number', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 103, 'name': 'split_id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 104, 'name': 'split_idx_start', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 105, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 106, 'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}, 'is_primary': True}
{'fi

In [ ]:
# Load collection into memory for search and query

client.load_collection(coll_name)
client.get_load_state(coll_name)

{'state': <LoadState: Loaded>}

In [ ]:
# Release collection from memory

client.release_collection(coll_name)
client.get_load_state(coll_name)

{'state': <LoadState: NotLoad>}

## Indexes

In [122]:
# List and describe indexes

index_name = client.list_indexes(coll_name)[0]
client.describe_index(coll_name, index_name)

{'index_type': 'AUTOINDEX',
 'metric_type': 'COSINE',
 'field_name': 'vector',
 'index_name': 'vector',
 'total_rows': 70,
 'indexed_rows': 70,
 'pending_index_rows': 0,
 'state': 'Finished'}

# Haystack-Zilliz Indexing Pipeline

In [32]:
from haystack import Pipeline, Document
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret

In [110]:
# Connect to Milvus client and create new collection

# file_names = ["Project Management Requirements Handbook.pdf"]

# document_store = MilvusDocumentStore(
#     collection_name = "HaystackCollection",
#     collection_description = "Test collection",
#     collection_properties = None,
#     connection_args = {
#         # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
#         "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
#         "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
#         "secure": True
#         },
#     consistency_level = "Strong", # Strong, Bounded, Eventually, Session
#     index_params = {
#         "index_type": "AUTOINDEX",
#         "metric_type": "COSINE",
#     },
#     search_params = {
#         "params": {
#             "level": 1
#         }
#     },
#     drop_old = True,
# )

# Connect to Milvus client and add to an existing collection

file_names = ["Lec1 Machine Learning Review.pdf"]

document_store = MilvusDocumentStore(
    collection_name = "HaystackCollection",
    # collection_description = "Test collection",
    # collection_properties = None,
    connection_args = {
        # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
        "secure": True
        },
    # consistency_level = "Strong", # Strong, Bounded, Eventually, Session
    # index_params = {
    #     "index_type": "AUTOINDEX",
    #     "metric_type": "COSINE",
    # },
    # search_params = {
    #     "params": {
    #         "level": 1
    #     }
    # },
    # drop_old=False,
)

In [ ]:
# Create indexing pipeline

pipe = Pipeline()

pipe.add_component("converter", PyPDFToDocument(extraction_mode="layout"))
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50))
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))

pipe.connect("converter", "cleaner")
pipe.connect("cleaner", "splitter")
pipe.connect("splitter", "embedder")
pipe.connect("embedder", "writer")

🚅 Components
  - converter: PyPDFToDocument
  - cleaner: DocumentCleaner
  - splitter: DocumentSplitter
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - converter.documents -> cleaner.documents (List[Document])
  - cleaner.documents -> splitter.documents (List[Document])
  - splitter.documents -> embedder.documents (List[Document])
  - embedder.documents -> writer.documents (List[Document])

In [111]:
# Run indexing pipeline

results = pipe.run({"converter": {"sources": file_names}}, include_outputs_from={"embedder"})

Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Document 1cd6c64adf832b9e6edbf14c8f71af72b6396e005e517111c870f7a7a8fe6231 has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 25666b0baad400b814f78d4ad0c6d1e2b95d171087b6bd88607e92879a6c616b has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document fdb129aa07cd821d5e0b412baef189785db89394ef3d62bc2d149534c5b1cf53 has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document a51b7670c9328cc33f268c7c7af77664a3385005663a131f69d6fc78341b4904 has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document a04c237152d7c60b81b7ebe1e1460043cfdfe745ef72f9851fa0072c1e7502fd has metadata f

In [115]:
document_store.count_documents()

70

In [113]:
# Sanity-check embedder output

idx = 0
print(results["embedder"]["documents"][idx])
print(f"id: {results["embedder"]["documents"][idx].id}")
for k, v in results["embedder"]["documents"][0].meta.items():
    print(f"{k}: {v}")

Document(id=1cd6c64adf832b9e6edbf14c8f71af72b6396e005e517111c870f7a7a8fe6231, content: 'Machine Learning ReviewIntroduction
to machine
learning 2What is Machine Learning? What is Machine...', meta: {'file_path': 'Lec1 Machine Learning Review.pdf', 'source_id': 'cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67', 'page_number': 1, 'split_id': 0, 'split_idx_start': 0, '_split_overlap': [{'doc_id': '25666b0baad400b814f78d4ad0c6d1e2b95d171087b6bd88607e92879a6c616b', 'range': (0, 90)}]}, embedding: vector of size 768)
id: 1cd6c64adf832b9e6edbf14c8f71af72b6396e005e517111c870f7a7a8fe6231
file_path: Lec1 Machine Learning Review.pdf
source_id: cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67
page_number: 1
split_id: 0
split_idx_start: 0
_split_overlap: [{'doc_id': '25666b0baad400b814f78d4ad0c6d1e2b95d171087b6bd88607e92879a6c616b', 'range': (0, 90)}]


In [114]:
# Describe the collection (schema)

col_name = document_store.collection_name

print(document_store.client.get_load_state(col_name))
col_description = document_store.client.describe_collection(col_name)
for k, v in col_description.items():
    if type(v) == list:
        for d in v:
            print(d)
    else:
        print(f"{k}: {v}")

{'state': <LoadState: Loaded>}
collection_name: HaystackCollection
auto_id: False
num_shards: 1
description: Test collection
{'field_id': 100, 'name': 'file_path', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 101, 'name': 'source_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 102, 'name': 'page_number', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 103, 'name': 'split_id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 104, 'name': 'split_idx_start', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}
{'field_id': 105, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 106, 'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}, 'is_primary': True}
{'field_id': 107, 'name': 'vector', 'description': '', 'type': <D

In [116]:
# Check how the Documents are written into the database (using MilvusClient.query)

expr = "id == {id}"
counter = 0
for doc in results["embedder"]["documents"]:
    filter_params = {"id": doc.id}
    res = document_store.client.query(
        collection_name = col_name,
        filter = expr,
        output_fields = ["*"],
        filter_params = filter_params
    )
    assert len(res) == 1, "Document IDs should map 1:1 to id in the collection"
    assert doc.id == res[0]["id"]
    assert doc.embedding == res[0]["vector"]
    assert doc.content == res[0]["text"]
    # assert doc.meta["source_id"] == res[0]["source_id"]
    # assert doc.meta["page_number"] == res[0]["page_number"]
    # assert doc.meta["split_id"] == res[0]["split_id"]
    # assert doc.meta["split_idx_start"] == res[0]["split_idx_start"]
    # assert doc.meta["file_path"] == res[0]["file_path"]

    for k, v in doc.meta.items():
        if k in res[0]:
            assert doc.meta[k] == res[0][k]

    counter += 1
    print(counter)
    # break

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30


In [117]:
# Check if doc.meta is the same for all documents (using DocumentStore.filter_documents)
doc = results["embedder"]["documents"][0]

# for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
#     expr = f"{field} == " + "{value}"
#     filter_params = {"value": doc.meta[field]}
#     res = document_store.client.query(
#         collection_name = col_name,
#         filter = expr,
#         output_fields = ["*"],
#         filter_params = filter_params
#     )
#     print(len(res))

for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
    filters = {"field": f"meta.{field}", "operator": "==", "value": doc.meta[field]}
    # print(filters)
    res = document_store.filter_documents(filters)
    print(len(res))

30
2
2
2
30


In [118]:
# Check doc.meta for all documents

for idx, doc in enumerate(results["embedder"]["documents"]):
    print(idx)
    for k, v in doc.meta.items():
        print(f"{k}: {v}")
    print("-"*30)

0
file_path: Lec1 Machine Learning Review.pdf
source_id: cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67
page_number: 1
split_id: 0
split_idx_start: 0
_split_overlap: [{'doc_id': '25666b0baad400b814f78d4ad0c6d1e2b95d171087b6bd88607e92879a6c616b', 'range': (0, 90)}]
------------------------------
1
file_path: Lec1 Machine Learning Review.pdf
source_id: cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67
page_number: 6
split_id: 1
split_idx_start: 833
_split_overlap: [{'doc_id': '1cd6c64adf832b9e6edbf14c8f71af72b6396e005e517111c870f7a7a8fe6231', 'range': (833, 923)}, {'doc_id': 'fdb129aa07cd821d5e0b412baef189785db89394ef3d62bc2d149534c5b1cf53', 'range': (0, 62)}]
------------------------------
2
file_path: Lec1 Machine Learning Review.pdf
source_id: cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67
page_number: 11
split_id: 2
split_idx_start: 1557
_split_overlap: [{'doc_id': '25666b0baad400b814f78d4ad0c6d1e2b95d171087b6bd88607e92879a6c616b', '

In [ ]:
filters = {"field": "meta.source_id", "operator": "==", "value": "cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67"}
filters = {"field": "meta.source_id", "operator": "==", "value": "8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e"}
res = document_store.filter_documents(filters)
print(len(res))
# res

40
